# Create WWTF Awards (Vienna Science and Technology Fund)

Creates Vienna Science and Technology Fund (Wiener Wissenschafts-, Forschungs- und
Technologiefonds, WWTF) awards from the fund's own public project database at
https://www.wwtf.at/funding/project-database/ . WWTF publishes a bulk export of the
whole database ("open as CSV document": project number, title, call, PI, home
institution, partners, start/end date, amount awarded in EUR, status, disciplines,
keywords); the script adds the abstract and co-PIs from each
project's detail page and the grant DOI from WWTF's Crossref deposits. **650 projects, calls 2003-2026 (start dates 2004-2026), EUR 307.3M after the one
amount exclusion below, 417 (64%) with a Crossref grant DOI** (local run 2026-10-01).

**Scope:** every WWTF programme is kept: the thematic calls (Life Sciences, ICT,
Mathematics, Cognitive Sciences, Environmental Systems Research, Social Sciences and
Humanities, Science for Creative Industries, Digital Humanism, DARE, COVID-19 Rapid
Response), Vienna Research Groups, Science Chairs, NEXT transfer projects, summer
schools, the Vienna Doctoral College on Digital Humanism, ME/CFS fellowships, and the
**Universitaets-Infrastrukturprogramm (UIP, 145 rows)**: research equipment and
infrastructure grants to Vienna universities (no PI; a few are teaching/arts-academy
infrastructure). UIP is kept as `funding_type = 'infrastructure'` and flagged in the
batch report (Kyle's 2026-10-01 scope rule: keep and flag when unsure).

**Prerequisites:**
- Run `scripts/local/wwtf_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/wwtf/wwtf_projects.parquet` (section 1.4 shrink guard).

**`funder_award_id` (section 2.1.1):** the WWTF project number (`LS17-012`,
`ICT19-045`, `VRG14-005`, `EI-COV20-016`, `UIP09-MUW`). WWTF deposits Crossref grant
DOIs (prefix 10.47379, e.g. 10.47379/LS17012) whose `award` is exactly this string;
all 417 deposits are in the CSV. Citation-side sample
(`openalex_awards_raw`, F4320321003, priority 0) is dominated by the same form
(`LS22-008` 222, `ICT15-113` 118, `MA09-024` 70 ...), so these rows upgrade the 417
`crossref_work` grant stubs (priority 1) and the matching acknowledgement shells.

**Amounts:** CSV `grants awarded` (EUR, German decimal comma `434810,16`; a lone
`93,796` is a thousands separator, confirmed: 2 x the academy's usual 46,898 UIP
share). The detail page's rounded `Funding volume` only fills gaps. Currency EUR.
**One source typo is NULLed here:** `UIP25-BOKU` is published as EUR 12,415,540
(CSV and detail page), ~100x every other UIP allocation (BOKU's UIP shares are
EUR 29k-206k; the other six 2025 UIP grants total EUR 1.6M), almost certainly
EUR 124,155.40 with the decimal comma lost. The raw value stays in `wwtf_raw`.

**DOI:** only WWTF's Crossref-registered grant DOIs (417). Newer projects print a
GrantID (`10.47379/LS25090`) on the detail page before it is registered (doi.org
404); that value is kept in the raw table as `detail_grant_id` only.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320321003`
- display_name: Vienna Science and Technology Fund
- ror_id / doi: from `openalex.funders.funders` (ror 01f9mc681, doi 10.13039/501100001821)

**Priority:** `530` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wwtf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/wwtf/wwtf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.wwtf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.wwtf_raw;

In [ ]:
%sql
SELECT funder_award_id, title, call, lead_name, lead_institution, csv_start_date, csv_end_date,
       amount, currency, grant_doi, landing_page_url
FROM openalex.awards.wwtf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321003 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321003;

## Step 2: Create WWTF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.wwtf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321003  -- Vienna Science and Technology Fund
),

people AS (
    -- PI(s) then co-PIs in page order -> investigator structs.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            from_json(people, 'ARRAY<STRUCT<role:STRING, name:STRING, given_name:STRING, family_name:STRING, institution:STRING>>'),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators
    FROM openalex.awards.wwtf_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    -- UIP25-BOKU: published EUR 12,415,540 is a source typo (see header); NULLed.
    CASE WHEN g.funder_award_id = 'UIP25-BOKU' AND TRY_CAST(g.amount AS DOUBLE) > 10000000 THEN NULL
         ELSE TRY_CAST(g.amount AS DOUBLE) END as amount,
    CASE WHEN g.funder_award_id = 'UIP25-BOKU' AND TRY_CAST(g.amount AS DOUBLE) > 10000000 THEN NULL
         WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE
        WHEN g.funder_award_id RLIKE '^UIP' THEN 'infrastructure'
        WHEN g.funder_award_id RLIKE '^(EI-)?VRG' THEN 'career_development'
        WHEN LOWER(g.call) LIKE '%fellowship%' THEN 'fellowship'
        WHEN LOWER(g.title) LIKE 'summer school%' OR g.funder_award_id RLIKE '^DCDH' THEN 'training'
        ELSE 'research'
    END as funding_type,
    NULLIF(TRIM(g.call), '') as funder_scheme,
    'wwtf_project_database' as provenance,
    TRY_TO_DATE(g.csv_start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.csv_end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.csv_start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.csv_end_date, 'yyyy-MM-dd')) as end_year,
    -- try_element_at: safe on empty arrays (section 2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CASE WHEN g.grant_doi IS NOT NULL THEN CONCAT('https://doi.org/', LOWER(g.grant_doi)) END as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.wwtf_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (section 2.1.1): every funder_award_id is a WWTF project number; 0 rows expected.
SELECT funder_award_id FROM openalex.awards.wwtf_awards
WHERE NOT funder_award_id RLIKE '^[A-Z]+(-[A-Z]+)?[0-9]{2}-[A-Za-z0-9_-]+$'
   OR funder_award_id LIKE '10.%';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wwtf_project_database' AND priority = 530;

-- Insert into openalex_awards_raw with priority.
-- Priority 530: direct-from-funder ingest of WWTF's own project database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 530 outranks the
-- crossref_work grant-DOI stubs (priority 1) and acknowledgement shells (priority 0)
-- that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    530 as priority
FROM openalex.awards.wwtf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_wwtf_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.wwtf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funding_type, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, doi
FROM openalex.awards.wwtf_awards LIMIT 20;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.wwtf_awards
GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.wwtf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.wwtf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.4a: titles should be distinct.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.wwtf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(doi) as has_doi,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.wwtf_awards;

In [ ]:
%sql
-- Overlap with the existing Crossref grant-DOI stubs (expect all 417 to share an id).
SELECT
    COUNT(*) as crossref_wwtf_rows,
    COUNT(t.id) as upgraded_by_wwtf_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.wwtf_awards t ON t.id = c.id
WHERE c.funder_id = 4320321003 AND c.provenance = 'crossref_work';

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'wwtf_project_database'
GROUP BY provenance, priority;